In [1]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

# 한글 폰트
_candidates = ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]
_available  = {f.name for f in fm.fontManager.ttflist}
_font       = next((f for f in _candidates if f in _available), None)
if _font:
    plt.rcParams["font.family"] = _font
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 100

# DOE 데이터셋 생성 (미니탭용)
원본 → 20분 리샘플 → SI 계산 → 3H Lag 정렬 → doe_dataset.csv 저장

In [2]:
# ── Steps 1~4: lag.ipynb와 동일 ──────────────────────────────────────────────

# Step 1. 원본 로드
df_raw = pd.read_csv(
    "MiningProcess_Flotation_Plant_Database.csv",
    decimal=",", thousands="."
)
df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw = df_raw.sort_values("date").reset_index(drop=True)

# Step 2. 타임스탬프 복원 (같은 시간대 행 순서 × 20초)
df_raw["_seq"]      = df_raw.groupby("date").cumcount()
df_raw["date_fine"] = df_raw["date"] + pd.to_timedelta(df_raw["_seq"] * 20, unit="s")
df_raw = df_raw.drop(columns=["date", "_seq"]).set_index("date_fine")
df_raw.index.name = "date"

# Step 3. 20분 리샘플 + 공백 구간 NaN 보존
df_20m = df_raw.resample("20min").mean().asfreq("20min")

# Step 4. SI 계산 (Rm, Jg 클리핑 → SI → 윈저라이징)
Feed_Fe = df_20m["% Iron Feed"]
Conc_Fe = df_20m["% Iron Concentrate"]
Feed_Si = df_20m["% Silica Feed"]
Conc_Si = df_20m["% Silica Concentrate"]

Y            = (Feed_Fe - 15.0) / (Conc_Fe - 15.0).replace(0, np.nan)
df_20m["Rm"] = (Y * (Conc_Fe / Feed_Fe) * 100).clip(0, 100)
df_20m["Jg"] = ((1 - Y * (Conc_Si / Feed_Si)) * 100).clip(0, 100)

si_inner     = (df_20m["Rm"] * df_20m["Jg"] /
                ((100.1 - df_20m["Rm"]) * (100.1 - df_20m["Jg"]))).clip(lower=0)
si_raw       = np.sqrt(si_inner)
si_99        = si_raw.quantile(0.99)
df_20m["SI"] = si_raw.clip(upper=si_99)

print(f"df_20m 준비 완료: {len(df_20m):,}행 × {df_20m.shape[1]}컬럼")
print(f"유효 SI 행: {df_20m['SI'].notna().sum():,}")
df_20m.head(9)

df_20m 준비 완료: 13,245행 × 26컬럼
유효 SI 행: 12,291


,% Iron Feed,% Silica Feed,Starch Flow,Amina Flow,Ore Pulp Flow,Ore Pulp pH,Ore Pulp Density,Flotation Column 01 Air Flow,Flotation Column 02 Air Flow,Flotation Column 03 Air Flow,...,Flotation Column 03 Level,Flotation Column 04 Level,Flotation Column 05 Level,Flotation Column 06 Level,Flotation Column 07 Level,% Iron Concentrate,% Silica Concentrate,Rm,Jg,SI
date,,,,,,,,,,,,,,,,,,,,,
2017-03-10 01:00:00,55.2,16.98,2938.926075,581.248383,397.369983,10.121225,1.722499,250.462133,250.004333,250.090317,...,451.181800,448.688100,453.905683,469.382450,445.126750,66.91,1.31,93.870034,94.025403,15.271599
2017-03-10 01:20:00,55.2,16.98,3430.846833,564.795883,399.646183,10.103617,1.723635,251.526000,250.644233,250.448883,...,449.635367,449.203333,448.870583,461.717867,450.121067,66.91,1.31,93.870034,94.025403,15.271599
2017-03-10 01:40:00,55.2,16.98,3113.155185,591.596778,399.298444,10.115857,1.743983,251.550241,250.007870,249.975370,...,450.621056,452.060130,465.578889,461.790315,456.996833,66.91,1.31,93.870034,94.025403,15.271599
2017-03-10 02:00:00,55.2,16.98,3373.192667,552.668550,400.123450,10.094875,1.687015,250.538067,249.996750,250.003767,...,451.058100,451.171000,455.568450,454.012850,455.502383,67.06,1.11,93.809401,94.952141,16.585029
2017-03-10 02:20:00,55.2,16.98,2980.648083,538.419100,399.442150,10.121930,1.668600,249.331400,249.750700,249.990233,...,448.989883,450.956117,440.500650,453.394250,446.950367,67.06,1.11,93.809401,94.952141,16.585029
2017-03-10 02:40:00,55.2,16.98,3045.928417,520.571333,400.049867,10.172420,1.647738,249.772300,250.894700,250.105950,...,450.195683,448.859300,450.099850,459.097483,451.710350,67.06,1.11,93.809401,94.952141,16.585029
2017-03-10 03:00:00,55.2,16.98,3377.862667,597.242533,396.742700,10.012403,1.745132,249.978533,250.148900,250.187617,...,452.108333,452.950083,452.553600,465.241050,446.444300,66.97,1.27,93.845739,94.214520,15.498414
2017-03-10 03:20:00,55.2,16.98,3522.168500,590.005883,399.648883,10.037700,1.728198,250.551667,249.984233,249.872683,...,450.522033,451.403000,452.407750,460.749967,454.883867,66.97,1.27,93.845739,94.214520,15.498414
2017-03-10 03:40:00,55.2,16.98,3538.417667,588.471817,399.899833,10.095107,1.724802,249.953783,250.179367,250.078750,...,450.075100,449.084383,448.441217,453.952917,449.562000,66.97,1.27,93.845739,94.214520,15.498414


In [3]:
LAG_STEPS = 9   # 9스텝 × 20분 = 180분 = 3H

# X 변수 컬럼 (SI 계산용 중간변수 Rm, Jg 제외, 결과 변수도 제외)
X_COLS = [
    "% Iron Feed", "% Silica Feed",
    "Starch Flow", "Amina Flow", "Ore Pulp Flow", "Ore Pulp pH", "Ore Pulp Density",
    "Flotation Column 01 Air Flow", "Flotation Column 02 Air Flow",
    "Flotation Column 03 Air Flow", "Flotation Column 04 Air Flow",
    "Flotation Column 05 Air Flow", "Flotation Column 06 Air Flow",
    "Flotation Column 07 Air Flow",
    "Flotation Column 01 Level", "Flotation Column 02 Level",
    "Flotation Column 03 Level", "Flotation Column 04 Level",
    "Flotation Column 05 Level", "Flotation Column 06 Level",
    "Flotation Column 07 Level",
]

# Lag 정렬: SI를 9스텝 앞으로 당기기
df_lag = df_20m[X_COLS].copy()
df_lag["SI_3H"] = df_20m["SI"].shift(-LAG_STEPS)

print(f"Lag 정렬 후 전체 행: {len(df_lag):,}")
print(f"  - 유효 쌍 (X & SI_3H 모두 존재): {df_lag.dropna().shape[0]:,}")
print(f"  - NaN 발생 (끝 {LAG_STEPS}행 + 공백 구간): {df_lag['SI_3H'].isna().sum():,}")


Lag 정렬 후 전체 행: 13,245
  - 유효 쌍 (X & SI_3H 모두 존재): 12,273
  - NaN 발생 (끝 9행 + 공백 구간): 963


In [4]:
# 과도 구간 제거 없이 NaN만 제거
df_clean = df_lag.dropna()

print("=== 데이터셋 요약 ===")
print(f"  원본 20분 행수     : {len(df_lag):,}")
print(f"  NaN 제거           : -{len(df_lag) - len(df_clean):,}")
print(f"  최종 유효 행수     : {len(df_clean):,}  ({len(df_clean)/len(df_lag)*100:.1f}% 보존)")
print(f"  컬럼 수            : {df_clean.shape[1]}  (X 21개 + SI_3H 1개)")
print()
print("=== SI_3H 기초 통계 ===")
display(df_clean["SI_3H"].describe().round(4))

# CSV 저장
output_path = "doe_dataset.csv"
df_clean.to_csv(output_path, encoding="utf-8-sig")
print(f"\n저장 완료: {output_path}")
display(df_clean.head(9))

=== 데이터셋 요약 ===
  원본 20분 행수     : 13,245
  NaN 제거           : -972
  최종 유효 행수     : 12,273  (92.7% 보존)
  컬럼 수            : 22  (X 21개 + SI_3H 1개)

=== SI_3H 기초 통계 ===


count    12273.0000
mean        13.6268
std          4.9562
min          0.0000
25%         10.2531
50%         12.6653
75%         15.6381
max         32.9454
Name: SI_3H, dtype: float64


저장 완료: doe_dataset.csv


,% Iron Feed,% Silica Feed,Starch Flow,Amina Flow,Ore Pulp Flow,Ore Pulp pH,Ore Pulp Density,Flotation Column 01 Air Flow,Flotation Column 02 Air Flow,Flotation Column 03 Air Flow,...,Flotation Column 06 Air Flow,Flotation Column 07 Air Flow,Flotation Column 01 Level,Flotation Column 02 Level,Flotation Column 03 Level,Flotation Column 04 Level,Flotation Column 05 Level,Flotation Column 06 Level,Flotation Column 07 Level,SI_3H
date,,,,,,,,,,,,,,,,,,,,,
2017-03-10 01:00:00,55.2,16.98,2938.926075,581.248383,397.369983,10.121225,1.722499,250.462133,250.004333,250.090317,...,250.482800,250.087833,450.558250,451.118908,451.181800,448.688100,453.905683,469.382450,445.126750,15.034222
2017-03-10 01:20:00,55.2,16.98,3430.846833,564.795883,399.646183,10.103617,1.723635,251.526000,250.644233,250.448883,...,251.925633,250.558500,448.369700,444.522725,449.635367,449.203333,448.870583,461.717867,450.121067,15.034222
2017-03-10 01:40:00,55.2,16.98,3113.155185,591.596778,399.298444,10.115857,1.743983,251.550241,250.007870,249.975370,...,251.295444,249.952667,452.427778,444.827463,450.621056,452.060130,465.578889,461.790315,456.996833,15.034222
2017-03-10 02:00:00,55.2,16.98,3373.192667,552.668550,400.123450,10.094875,1.687015,250.538067,249.996750,250.003767,...,249.621150,249.861167,448.581567,449.727283,451.058100,451.171000,455.568450,454.012850,455.502383,15.197703
2017-03-10 02:20:00,55.2,16.98,2980.648083,538.419100,399.442150,10.121930,1.668600,249.331400,249.750700,249.990233,...,250.112783,249.850933,450.240500,459.578067,448.989883,450.956117,440.500650,453.394250,446.950367,15.197703
2017-03-10 02:40:00,55.2,16.98,3045.928417,520.571333,400.049867,10.172420,1.647738,249.772300,250.894700,250.105950,...,249.994550,249.980617,449.298017,441.442717,450.195683,448.859300,450.099850,459.097483,451.710350,15.197703
2017-03-10 03:00:00,55.2,16.98,3377.862667,597.242533,396.742700,10.012403,1.745132,249.978533,250.148900,250.187617,...,250.033233,250.300533,448.634317,453.230117,452.108333,452.950083,452.553600,465.241050,446.444300,16.368152
2017-03-10 03:20:00,55.2,16.98,3522.168500,590.005883,399.648883,10.037700,1.728198,250.551667,249.984233,249.872683,...,250.449850,250.351150,449.578333,449.661850,450.522033,451.403000,452.407750,460.749967,454.883867,16.368152
2017-03-10 03:40:00,55.2,16.98,3538.417667,588.471817,399.899833,10.095107,1.724802,249.953783,250.179367,250.078750,...,250.243400,250.800867,451.705983,449.714167,450.075100,449.084383,448.441217,453.952917,449.562000,16.368152
